## Clean NIFD data and select latest visits

Created by: Sahana 
Modifeid by: Varuna

Date: 05/07/2025

In [1]:
import pandas as pd
import numpy as np
import json

In [2]:
directory_path = "/projectnb/vkolagrp/skowshik/foundation_adrd/adrd-foundation-model/data/nifd"

In [3]:
with open(f"{directory_path}/NIFD_dictionary.json", 'r') as json_file:
    data_dict = json.load(json_file)

In [4]:
nifd = pd.read_csv('/projectnb/vkolagrp/datasets/NIFD/csv/raw/NIFD_Clinical_Data_20200724_updated.csv')
nifd.head()

,LONI_ID,CLINICAL_LINKDATE,DX,SITE,VISIT_NUMBER,DOB,GENDER,EDUCATION,RACE,CDR_DCDATE,...,RSMS_FTDCHBEH,RSMS_FTDADBEH,RSMS_FTDLYING,RSMS_FTDGOODF,RSMS_FTDREGUL,RSMS_FTDSMSCR,RSMS_FTDSPSCR,MDEXAM_DCDATE,MDEXAM_UHDRS,MDEXAM_UPDRS
0,1_S_0190,2/14/14,CON,UCSF,1,8/11/61,1,20,1,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2/14/14,1.0,2.0
1,1_S_0190,8/29/14,CON,UCSF,2,8/11/61,1,20,1,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,8/29/14,1.0,0.0
2,1_S_0190,7/17/15,CON,UCSF,3,8/11/61,1,20,1,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,1_S_0231,3/14/13,CON,UCSF,2,8/25/40,1,20,1,4/3/13,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3/14/13,0.0,0.0
4,1_S_0231,8/26/15,CON,UCSF,3,8/25/40,1,20,1,4/17/15,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [5]:
nifd["EDUCATION"] = nifd["EDUCATION"].replace(99, np.nan)

In [6]:
nifd['CLINICAL_LINKDATE']

0       2/14/14
1       8/29/14
2       7/17/15
3       3/14/13
4       8/26/15
         ...   
1016    1/19/16
1017    8/13/15
1018    7/29/14
1019    7/18/14
1020    1/15/15
Name: CLINICAL_LINKDATE, Length: 1021, dtype: object

In [7]:
dt = pd.to_datetime(nifd['CLINICAL_LINKDATE'], errors='coerce', yearfirst=False)
dt

/scratch/ipykernel_3563192/663156243.py:1: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  dt = pd.to_datetime(nifd['CLINICAL_LINKDATE'], errors='coerce', yearfirst=False)


0      2014-02-14
1      2014-08-29
2      2015-07-17
3      2013-03-14
4      2015-08-26
          ...    
1016   2016-01-19
1017   2015-08-13
1018   2014-07-29
1019   2014-07-18
1020   2015-01-15
Name: CLINICAL_LINKDATE, Length: 1021, dtype: datetime64[ns]

In [8]:
# Convert DOB and CLINICAL_LINKDATE to datetime, handling ambiguous DOB years
def parse_dob(dob_str):
    try:
        # Try parsing as 4-digit year first
        dt = pd.to_datetime(dob_str, errors='coerce', yearfirst=False)
        if pd.isnull(dt):
            return None
        # If year is in the future (e.g., 2061), assume it's 19xx not 20xx
        if dt.year > 2020:
            dt = dt.replace(year=dt.year - 100)
        return dt
    except Exception:
        return None

nifd['DOB_convert'] = nifd['DOB'].apply(parse_dob)
# nifd_latest['DOB_convert'] = pd.to_datetime(nifd_latest['DOB'], errors='coerce')
nifd['CLINICAL_LINKDATE_convert'] = pd.to_datetime(nifd['CLINICAL_LINKDATE'], errors='coerce')

# Ensure CLINICAL_LINKDATE is always after DOB
# mask = (nifd_latest['CLINICAL_LINKDATE'] > nifd_latest['DOB'])
# nifd_latest.loc[~mask, 'DOB'] = pd.NaT  # Set impossible DOBs to NaT

# Calculate age in years at the time of visit
nifd['AGE'] = (nifd['CLINICAL_LINKDATE_convert'] - nifd['DOB_convert']).dt.days / 365.25


/scratch/ipykernel_3563192/2738008283.py:17: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  nifd['CLINICAL_LINKDATE_convert'] = pd.to_datetime(nifd['CLINICAL_LINKDATE'], errors='coerce')


In [9]:
# print(nifd['DOB_convert'].min())
# print(nifd['DOB_convert'].max())
# print(nifd['AGE'].min())
# print(nifd['AGE'].max())

In [10]:
nifd.shape

(1021, 126)

# Get latest visit

In [11]:
def get_latest_visits(data):
    result = data.sort_values(by=['LONI_ID', 'VISIT_NUMBER'], ascending=[True, False])
    result = result.drop_duplicates(subset='LONI_ID', keep='first').reset_index(drop=True)
    return result

nifd_latest = get_latest_visits(nifd)
nifd_latest['COHORT'] = "NIFD"

In [12]:
nifd_latest[nifd_latest['DX'].isna()]

,LONI_ID,CLINICAL_LINKDATE,DX,SITE,VISIT_NUMBER,DOB,GENDER,EDUCATION,RACE,CDR_DCDATE,...,RSMS_FTDREGUL,RSMS_FTDSMSCR,RSMS_FTDSPSCR,MDEXAM_DCDATE,MDEXAM_UHDRS,MDEXAM_UPDRS,DOB_convert,CLINICAL_LINKDATE_convert,AGE,COHORT
307,2_S_0011,8/24/11,NaN,MAYO,1,8/27/50,1,12.0,1,8/24/11,...,NaN,NaN,NaN,NaN,NaN,NaN,1950-08-27,2011-08-24,60.991102,NIFD
311,2_S_0015,3/28/12,NaN,MAYO,1,8/5/42,1,16.0,1,3/28/12,...,NaN,NaN,NaN,3/28/12,8.0,14.0,1942-08-05,2012-03-28,69.645448,NIFD
320,2_S_0024,9/26/12,NaN,MAYO,1,8/14/44,1,14.0,1,9/26/12,...,NaN,NaN,NaN,9/26/12,-5.0,-5.0,1944-08-14,2012-09-26,68.117728,NIFD
321,2_S_0025,9/26/12,NaN,MAYO,1,2/5/46,2,13.0,1,9/26/12,...,NaN,NaN,NaN,9/26/12,-5.0,-5.0,1946-02-05,2012-09-26,66.639288,NIFD
322,2_S_0026,10/2/12,NaN,MAYO,1,5/3/56,1,NaN,99,10/2/12,...,NaN,NaN,NaN,10/2/12,-5.0,-5.0,1956-05-03,2012-10-02,56.416153,NIFD
324,2_S_0028,1/23/13,NaN,MAYO,1,4/28/42,1,20.0,1,1/23/13,...,NaN,NaN,NaN,1/23/13,-5.0,-5.0,1942-04-28,2013-01-23,70.740589,NIFD


# Define Form IDs

In [13]:
form = {
    'A1': 'Subject Demographics',
    'B3': "Unified Parkinson's Disease Rating Scale (UPDRS)",
    'B4': 'Clinical Dementia Rating (CDR)',
    'B5': 'Neuropsychiatric Inventory',
    'B6': 'Geriatric Depression Scale (GDS)',
    'B7': 'Activities of Daily Living',
    'C': 'Neuropsychological Battery Summary Scores',
    'D1': 'Clinician Diagnosis',
    'I': 'MRI Report'
}

In [14]:
skip_cols = []
for k, v in data_dict.items():
    if v['FORM ID'] in ['U1', 'B4', 'D1'] or v['Type'].lower() == "datetime":
        skip_cols.append(k)

# Map tabular data to JSON

In [15]:
def create_mapped_json(row):
    result = {}
    
    for column, value in row.items():
        if column in skip_cols:
            continue
            
        # skip if nan
        if pd.isna(value) or value is None:
            continue
        # convert to number to check if value's negative
        try:
            if float(value) < 0:
                continue  
        except (ValueError, TypeError):
            pass  # Not a number, continue processing
        
        if column not in data_dict:
            continue
        
        form_desc = form[data_dict[column]['FORM ID']]
        if form_desc not in result:
            result[form_desc] = {}
            
        # convert value to appropriate type if specified in dictionary
        if column in data_dict and 'Type' in data_dict[column]:
            try:
                data_type = data_dict[column]['Type']
                if data_type == 'int' or data_type == 'integer':
                    value = int(float(value))
                elif data_type == 'float' or data_type == 'number':
                    value = float(value)
                elif data_type == 'str' or data_type == 'string':
                    # Keep as string, but we already checked for negative values above
                    value = str(value)
                elif data_type == 'bool' or data_type == 'boolean':
                    value = bool(value)
            except (ValueError, TypeError):
                # If conversion fails, skip this value
                continue
            
        if column in data_dict:
            description = data_dict[column]['Description']
            
            # Check if the column has a values key in the dictionary
            if 'Values' in data_dict[column]:
                # print(column, value)
                if str(value) in data_dict[column]['Values']:
                    mapped_value = data_dict[column]['Values'][str(value)]
                    # print(column, value, mapped_value)
                    if mapped_value:
                        result[form_desc][description] = mapped_value
                    # print(result[form_desc])
                # Skip this field if value doesn't match any in the dictionary
                else:
                    continue
            # For columns without Values key in dictionary, add the value directly
            else:
                result[form_desc][description] = value
                
        # add T1 findings if available:
        if 'brain_findings_summary' in row and pd.notna(row['brain_findings_summary']):
            gamlss_description = "A Generalized Additive Model for Location, Scale, and Shape (GAMLSS) was fitted to FastSurfer-derived regional brain volumes acquired from 2,087 healthy controls, following the Brain charts for the human lifespan methodology published in Nature (2022). Data were aggregated from five independent cohorts, and the analysis included region-of-interest (ROI) volumes for the medial and lateral temporal lobes, medial and lateral parietal lobes, frontal, occipital lobes, as well as the inferior lateral ventricles and hippocampus. To ensure a robust normative reference, participants with any known neurodegenerative disease were excluded and subjects with extreme brain volumes (>3.5 absolute z-scores) were excluded during quality control prior to model fitting. The Brain Charts pipeline was implemented to model age- and sex-specific normative centile curves for each ROI. Centile scores (scaled 0–1) were computed for both healthy controls and patient samples. Abnormality was defined based on a one-sided z-score threshold: mild atrophy/enlargement for scores corresponding to 1 standard deviation (z=1), moderate for 1.5 SD (z=1.5), and severe for 2 SD (z=2). For all lobar ROIs, lower centiles indicated greater atrophy, while for ventricular ROIs, higher centiles indicated enlargement (i.e., abnormality). The resulting findings are as follows: ",

            # result["T1 MRI findings"] = gamlss_description + str(row['brain_findings_summary'])
            result["MRI Report"] = {
                "MRI features extraction procedure" : gamlss_description,
                "T1 MRI Findings" : str(row['brain_findings_summary'])
            }
            
    result = {k: v for k, v in result.items() if v}
    key_order = [k for k in form.values() if k in result]
    result = {k: result[k] for k in key_order if k in result}
                
    return json.dumps(result)

In [16]:
row1=nifd_latest.iloc[0]

In [17]:
row1['DX']

'CON'

In [18]:
def create_diag_summary(row):
    if 'DX' not in row or pd.isna(row['DX']):
        return json.dumps({})
    
    field_info = data_dict['DX']
    description = field_info['Description']
    value = row['DX']
    
    # Apply value mapping if available
    if 'Values' in field_info:
        if value in field_info['Values']:
            mapped_value = field_info['Values'][value]
            # print(f"Found mapping: {value} -> {mapped_value}")
        else:
            mapped_value = value
            print(f"No mapping found for {value}, using original value")
    else:
        mapped_value = value
        print("No Values field found")
    
    result = {description: mapped_value}
    return json.dumps(result)

In [19]:
nifd_latest['DX'].value_counts()

DX
CON                136
BV                  77
PNFA                40
SV                  39
PATIENT (OTHER)     39
L_SD                 4
Name: count, dtype: int64

In [20]:
nifd_latest[nifd_latest['DX'].isna()]

,LONI_ID,CLINICAL_LINKDATE,DX,SITE,VISIT_NUMBER,DOB,GENDER,EDUCATION,RACE,CDR_DCDATE,...,RSMS_FTDREGUL,RSMS_FTDSMSCR,RSMS_FTDSPSCR,MDEXAM_DCDATE,MDEXAM_UHDRS,MDEXAM_UPDRS,DOB_convert,CLINICAL_LINKDATE_convert,AGE,COHORT
307,2_S_0011,8/24/11,NaN,MAYO,1,8/27/50,1,12.0,1,8/24/11,...,NaN,NaN,NaN,NaN,NaN,NaN,1950-08-27,2011-08-24,60.991102,NIFD
311,2_S_0015,3/28/12,NaN,MAYO,1,8/5/42,1,16.0,1,3/28/12,...,NaN,NaN,NaN,3/28/12,8.0,14.0,1942-08-05,2012-03-28,69.645448,NIFD
320,2_S_0024,9/26/12,NaN,MAYO,1,8/14/44,1,14.0,1,9/26/12,...,NaN,NaN,NaN,9/26/12,-5.0,-5.0,1944-08-14,2012-09-26,68.117728,NIFD
321,2_S_0025,9/26/12,NaN,MAYO,1,2/5/46,2,13.0,1,9/26/12,...,NaN,NaN,NaN,9/26/12,-5.0,-5.0,1946-02-05,2012-09-26,66.639288,NIFD
322,2_S_0026,10/2/12,NaN,MAYO,1,5/3/56,1,NaN,99,10/2/12,...,NaN,NaN,NaN,10/2/12,-5.0,-5.0,1956-05-03,2012-10-02,56.416153,NIFD
324,2_S_0028,1/23/13,NaN,MAYO,1,4/28/42,1,20.0,1,1/23/13,...,NaN,NaN,NaN,1/23/13,-5.0,-5.0,1942-04-28,2013-01-23,70.740589,NIFD


In [21]:
nifd_latest['patient_summary'] = nifd_latest.apply(create_mapped_json, axis=1)
nifd_latest['diag_summary'] = nifd_latest.apply(create_diag_summary, axis=1)

In [22]:
nifd_latest['diag_summary'].value_counts()

diag_summary
{"Diagnosis": "Healthy Control"}                       136
{"Diagnosis": "Behavioral Variant FTD"}                 77
{"Diagnosis": "Other Diagnosis"}                        43
{"Diagnosis": "Progressive Non-Fluent Aphasia FTD"}     40
{"Diagnosis": "Semantic Variant FTD"}                   39
{}                                                       6
Name: count, dtype: int64

In [23]:
nifd_latest[nifd_latest['RACE'] == 99].iloc[0]['patient_summary']

'{"Subject Demographics": {"Gender": "Male", "Years of Education": 15, "Age": 62}, "Unified Parkinson\'s Disease Rating Scale (UPDRS)": {"MD Exam - Unified Parkinson\'s Disease Rating Scale": 3}, "Activities of Daily Living": {"Schwab and England Activities of Daily Living - Score": 20}}'

In [25]:
json.loads(nifd_latest.iloc[2]['patient_summary'])

{'Subject Demographics': {'Gender': 'Male',
  'Years of Education': 12,
  'Race': 'White',
  'Age': 68},
 "Unified Parkinson's Disease Rating Scale (UPDRS)": {"MD Exam - Unified Huntington's Disease Rating Scale": 12,
  "MD Exam - Unified Parkinson's Disease Rating Scale": 14},
 'Neuropsychiatric Inventory': {'Neuropsychiatric Inventory - Q Form - Delusions': 'No',
  'Neuropsychiatric Inventory - Q Form - Hallucination Distress': 0,
  'Neuropsychiatric Inventory - Q Form - Agitation': 'No',
  'Neuropsychiatric Inventory - Q Form - Depression': 'No',
  'Neuropsychiatric Inventory - Q Form - Anxiety': 'No',
  'Neuropsychiatric Inventory - Q Form - Elation/Euphoria': 'Yes',
  'Neuropsychiatric Inventory - Q Form - Elation/Euphoria Severity': 2,
  'Neuropsychiatric Inventory - Q Form - Apathy/Indifference': 'Yes',
  'Neuropsychiatric Inventory - Q Form - Apathy/Indifference Severity': 3,
  'Neuropsychiatric Inventory - Q Form - Disinhibition': 'Yes',
  'Neuropsychiatric Inventory - Q Form 

In [26]:
nifd_latest

,LONI_ID,CLINICAL_LINKDATE,DX,SITE,VISIT_NUMBER,DOB,GENDER,EDUCATION,RACE,CDR_DCDATE,...,RSMS_FTDSPSCR,MDEXAM_DCDATE,MDEXAM_UHDRS,MDEXAM_UPDRS,DOB_convert,CLINICAL_LINKDATE_convert,AGE,COHORT,patient_summary,diag_summary
0,1_S_0001,8/27/14,CON,UCSF,7,12/9/45,1,15.0,1,NaN,...,NaN,NaN,NaN,NaN,1945-12-09,2014-08-27,68.714579,NIFD,"{""Subject Demographics"": {""Gender"": ""Male"", ""Y...","{""Diagnosis"": ""Healthy Control""}"
1,1_S_0002,1/24/12,BV,UCSF,3,12/28/54,2,13.0,1,1/24/12,...,NaN,NaN,NaN,NaN,1954-12-28,2012-01-24,57.073238,NIFD,"{""Subject Demographics"": {""Gender"": ""Female"", ...","{""Diagnosis"": ""Behavioral Variant FTD""}"
2,1_S_0003,8/20/12,SV,UCSF,4,6/8/44,1,12.0,1,8/15/12,...,NaN,8/15/12,12.0,14.0,1944-06-08,2012-08-20,68.199863,NIFD,"{""Subject Demographics"": {""Gender"": ""Male"", ""Y...","{""Diagnosis"": ""Semantic Variant FTD""}"
3,1_S_0005,8/26/14,BV,UCSF,6,11/22/51,1,18.0,1,8/26/14,...,12.0,8/26/14,-5.0,15.0,1951-11-22,2014-08-26,62.759754,NIFD,"{""Subject Demographics"": {""Gender"": ""Male"", ""Y...","{""Diagnosis"": ""Behavioral Variant FTD""}"
4,1_S_0006,7/15/11,BV,UCSF,4,7/24/48,1,15.0,99,NaN,...,NaN,7/15/11,-5.0,3.0,1948-07-24,2011-07-15,62.973306,NIFD,"{""Subject Demographics"": {""Gender"": ""Male"", ""Y...","{""Diagnosis"": ""Behavioral Variant FTD""}"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
336,3_S_0008,2/21/14,PNFA,MGH,4,1/1/37,1,14.0,1,2/21/14,...,NaN,2/21/14,38.0,33.0,1937-01-01,2014-02-21,77.138946,NIFD,"{""Subject Demographics"": {""Gender"": ""Male"", ""Y...","{""Diagnosis"": ""Progressive Non-Fluent Aphasia ..."
337,3_S_0009,8/16/13,BV,MGH,3,1/1/57,1,12.0,1,8/16/13,...,NaN,8/16/13,-5.0,-5.0,1957-01-01,2013-08-16,56.621492,NIFD,"{""Subject Demographics"": {""Gender"": ""Male"", ""Y...","{""Diagnosis"": ""Behavioral Variant FTD""}"
338,3_S_0010,3/12/14,SV,MGH,4,1/1/51,1,16.0,1,3/12/14,...,NaN,3/24/14,-5.0,-5.0,1951-01-01,2014-03-12,63.192334,NIFD,"{""Subject Demographics"": {""Gender"": ""Male"", ""Y...","{""Diagnosis"": ""Semantic Variant FTD""}"
339,3_S_0011,4/7/14,SV,MGH,4,1/1/47,1,13.0,1,4/7/14,...,NaN,4/7/14,-5.0,-5.0,1947-01-01,2014-04-07,67.263518,NIFD,"{""Subject Demographics"": {""Gender"": ""Male"", ""Y...","{""Diagnosis"": ""Semantic Variant FTD""}"


In [27]:
nifd_latest.to_csv(f'{directory_path}/NIFD_wjson.csv', index=False)

In [28]:
nifd_latest_filtered = nifd_latest[nifd_latest['DX'].isin(['CON', 'BV', 'PNFA', 'SV'])].reset_index(drop=True)

In [29]:
len(nifd_latest_filtered)

292

In [30]:
nifd_latest_filtered[nifd_latest_filtered['DX'].isna()]

,LONI_ID,CLINICAL_LINKDATE,DX,SITE,VISIT_NUMBER,DOB,GENDER,EDUCATION,RACE,CDR_DCDATE,...,RSMS_FTDSPSCR,MDEXAM_DCDATE,MDEXAM_UHDRS,MDEXAM_UPDRS,DOB_convert,CLINICAL_LINKDATE_convert,AGE,COHORT,patient_summary,diag_summary


In [31]:
nifd_latest_filtered.to_csv(f'{directory_path}/test.csv', index=False)

In [32]:
directory_path

'/projectnb/vkolagrp/skowshik/foundation_adrd/adrd-foundation-model/data/nifd'